In [ ]:
MIRBI=10×SWIR2−9.8×SWIR1+2 

In [8]:

import arcpy
from arcpy.sa import *
import os
arcpy.env.overwriteOutput = True  # <-- Add this line
import math


In [9]:
arcpy.CheckOutExtension("Spatial")


'CheckedOut'

In [22]:
#first check if all files in the same spatial coord system


input_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\Landsat8_BR"

# Dictionary to track different spatial references
spatial_refs = {}

# Process each raster
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Process only TIFF files
        raster_path = os.path.join(input_folder, raster_file)
        
        # Get raster spatial reference
        desc = arcpy.Describe(raster_path)
        spatial_ref = desc.spatialReference.name  # Get name of spatial reference
        
        # Store in dictionary (count occurrences)
        if spatial_ref not in spatial_refs:
            spatial_refs[spatial_ref] = []
        spatial_refs[spatial_ref].append(raster_file)

# Print results
print("\n📌 Spatial Reference Check:")
if len(spatial_refs) == 1:
    print(f"✅ All rasters are in the same spatial reference: {list(spatial_refs.keys())[0]}")
else:
    print("⚠️ Different spatial references detected!")
    for ref, files in spatial_refs.items():
        print(f"🔹 {ref}: {len(files)} rasters")
        for f in files[:5]:  # Show up to 5 files for each spatial ref
            print(f"    - {f}")


📌 Spatial Reference Check:
✅ All rasters are in the same spatial reference: WGS_1984_UTM_Zone_17N


In [ ]:
arcpy.management.ProjectRaster(
    in_raster=r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_Cro\Landsat8_Cro\Landsat_LC08_015036_20210121.tif",
    out_raster=r"F:\remote sensing fire detection\VFT remote sensing fire detection\VFT remote sensing fire detection.gdb\Landsat_LC08_0_ProjectRaster_test2",
    out_coor_system='PROJCS["WGS_1984_UTM_Zone_18N",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-75.0],PARAMETER["Scale_Factor",0.9996],PARAMETER["Latitude_Of_Origin",0.0],UNIT["Meter",1.0]]',
    resampling_type="BILINEAR",
    cell_size="30 30",
    geographic_transform=None,
    Registration_Point=None,
    in_coor_system='PROJCS["WGS_1984_UTM_Zone_17N",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-81.0],PARAMETER["Scale_Factor",0.9996],PARAMETER["Latitude_Of_Origin",0.0],UNIT["Meter",1.0]]',
    vertical="NO_VERTICAL"
)

In [13]:
# if different spatial reference, convert
input_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\Cro_additional_2"
output_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected"

target_spatial_ref = arcpy.SpatialReference(32618)  # UTM Zone 18N (Modify if needed)   # UTM Zone 17N (EPSG: 32617)  

# Process each raster
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Process only TIFF files
        raster_path = os.path.join(input_folder, raster_file)
        
        # Get raster properties
        desc = arcpy.Describe(raster_path)
        current_spatial_ref = desc.spatialReference
        
        # Check if reprojection is needed
        if current_spatial_ref.factoryCode != target_spatial_ref.factoryCode:
            output_path = os.path.join(output_folder, f"Reprojected_{raster_file}")
            
            # Reproject raster
            arcpy.management.ProjectRaster(
                raster_path, output_path, target_spatial_ref,
                "BILINEAR", 30  # Resampling method & cell size (modify if needed)
            )
            print(f"✅ Reprojected: {raster_file} → {output_path}")
        else:
            output_path = os.path.join(output_folder, raster_file)  
            arcpy.CopyRaster_management(raster_path, output_path)
            print(f"✔️ Already in target CRS: {raster_file} ➞ Copied to {output_path}")
            
print("fin")
      
# go back to check for consistent projection

✔️ Already in target CRS: Landsat_LT05_014036_19900415.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19900415.tif
✔️ Already in target CRS: Landsat_LT05_014036_19900501.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19900501.tif
✔️ Already in target CRS: Landsat_LT05_014036_19900517.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19900517.tif
✔️ Already in target CRS: Landsat_LT05_014036_19900602.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19900602.tif
✔️ Already in target CRS: Landsat_LT05_014036_19900704.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19900704.tif


✔️ Already in target CRS: Landsat_LT05_014036_19950208.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19950208.tif
✔️ Already in target CRS: Landsat_LT05_014036_19950224.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19950224.tif
✔️ Already in target CRS: Landsat_LT05_014036_19950312.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19950312.tif
✔️ Already in target CRS: Landsat_LT05_014036_19950328.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19950328.tif
✔️ Already in target CRS: Landsat_LT05_014036_19950413.tif ➞ Copied to F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Landsat_LT05_014036_19950413.tif


✅ Reprojected: Landsat_LT05_015036_19960218.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960218.tif
✅ Reprojected: Landsat_LT05_015036_19960305.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960305.tif
✅ Reprojected: Landsat_LT05_015036_19960321.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960321.tif
✅ Reprojected: Landsat_LT05_015036_19960422.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960422.tif
✅ Reprojected: Landsat_LT05_015036_19960508.tif → F:\remote sensing fire detection\VFT remote sensing fire detection\Cro_additional_2\reprojected\Reprojected_Landsat_LT05_015036_19960508.tif
✅ Reprojected: Landsat_LT05_015036_19960524.t

In [8]:

### get unqiue values in a pixel


input_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat5_BR\Landsat5_BR"
B1 = arcpy.Point(663035,3890743)

unique_values_B1 = set()
#unique_values_bi = set()

# Loop through rasters
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):
        raster_path = os.path.join(input_folder, raster_file)
        print(f"Processing {raster_file}...")

        # Extract band 3 (assumed to be QA_PIXEL band)
        qa_band = arcpy.sa.ExtractBand(raster_path, 6)

        # Extract pixel values at GSP points
        unique_values_B1 = arcpy.GetCellValue_management(qa_band, f"{B1.X} {B1.Y}")

        # Get raw values
        val_B1 = int(value_B1.getOutput(0))

        # Add to sets
        unique_values_B1.add(val_B1)

# Print unique values
print("\n📌 Unique QA_PIXEL values at GSP_LI:")
print(sorted(unique_values_B1))

Processing Landsat_LT05_015036_20070115.tif...


AttributeError: 'float' object has no attribute 'getOutput'

In [10]:
# filter for cloud free images

input_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\Landsat8_BR"
output_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\filtered"

# Define the exact cloud-free QA value and coordinates (depends on sites)
CLOUD_FREE_QA = 21824  #for landsat 8-9
#CLOUD_FREE_QA = 5440 # for landsat 5 bit flag 6 -- clear
gsp_li = arcpy.Point(749137, 3774536)  
gsp_bi = arcpy.Point(748942, 3774918) 
IA = arcpy.Point(288256,3830171)
CH = arcpy.Point(328503,3848824)
CM = arcpy.Point(319321,3848871)
B1 = arcpy.Point(663035,3890743)
B2 = arcpy.Point(664053,3890735)
ME = arcpy.Point(273569,3836308)

# Process each raster
#for raster_file in os.listdir(input_folder):
     #if raster_file.endswith(".tif") and not raster_file.endswith(".tif.xml"):  # Only process .tif files
        #raster_path = os.path.join(input_folder, raster_file)
#for raster_file in os.listdir(input_folder):
    #print(f"Scanning: {raster_file}")  # 👈 debug
   # if raster_file.lower().endswith(".tif") and ".xml" not in raster_file.lower():
        # Extract QA_PIXEL band (assuming it’s band 3)
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):
        raster_path = os.path.join(input_folder, raster_file)
        
        qa_band = arcpy.sa.ExtractBand(raster_path, 6) # band number, modify if needed

        # Extract pixel values at the two points
        #value_IA = arcpy.GetCellValue_management(qa_band, f"{IA.X} {IA.Y}")
        #value_CH = arcpy.GetCellValue_management(qa_band, f"{CH.X} {CH.Y}")
        #value_CM = arcpy.GetCellValue_management(qa_band, f"{CM.X} {CM.Y}")
        #value_gsp_li = arcpy.GetCellValue_management(qa_band, f"{gsp_li.X} {gsp_li.Y}")
        #value_gsp_bi = arcpy.GetCellValue_management(qa_band, f"{gsp_bi.X} {gsp_bi.Y}")
        value_B1_raw = arcpy.GetCellValue_management(qa_band, f"{B1.X} {B1.Y}").getOutput(0)
        value_B2_raw = arcpy.GetCellValue_management(qa_band, f"{B2.X} {B2.Y}").getOutput(0)
    

        if value_B1_raw == "NoData" or value_B2_raw == "NoData":
            print(f"❌ Skipped {raster_file} (NoData at either)")
            continue  # Skip this raster and move to the next one
        #import math
        value_B1 = float(value_B1_raw)
        value_B2 = float(value_B2_raw)
      
        if math.isnan(value_B1) or math.isnan(value_B2):
            print(f"❌ Skipped {raster_file} (NaN at either)")
            continue
# Keep only files where both values are exactly 21824
        if int(value_B1) == CLOUD_FREE_QA and int(value_B2) == CLOUD_FREE_QA:  # and value_CM == CLOUD_FREE_QA:
            output_path = os.path.join(output_folder, raster_file)
            arcpy.CopyRaster_management(raster_path, output_path)
    
            print(f"✅ Kept {raster_file} (cloud free)")
        else:
            print(f"❌ Skipped {raster_file} (cloud)")

print("🎉 Filtering complete!")

✅ Kept Landsat_LC08_015036_20130402.tif (cloud free)
✅ Kept Landsat_LC08_016035_20130717.tif (cloud free)
❌ Skipped Landsat_LC08_016035_20130818.tif (cloud)
❌ Skipped Landsat_LC08_016035_20130903.tif (cloud)
❌ Skipped Landsat_LC08_016035_20130919.tif (cloud)
✅ Kept Landsat_LC08_016035_20131005.tif (cloud free)
❌ Skipped Landsat_LC08_016035_20131021.tif (cloud)
✅ Kept Landsat_LC08_016035_20131106.tif (cloud free)
❌ Skipped Landsat_LC08_016035_20131122.tif (cloud)
❌ Skipped Landsat_LC08_016035_20131224.tif (cloud)
✅ Kept Landsat_LC08_016035_20140109.tif (cloud free)
❌ Skipped Landsat_LC08_016035_20140125.tif (cloud)
❌ Skipped Landsat_LC08_016035_20140226.tif (cloud)
✅ Kept Landsat_LC08_016035_20140314.tif (cloud free)
❌ Skipped Landsat_LC08_016035_20140330.tif (cloud)
❌ Skipped Landsat_LC08_016035_20140501.tif (cloud)
✅ Kept Landsat_LC08_016035_20140517.tif (cloud free)
✅ Kept Landsat_LC08_016035_20140602.tif (cloud free)
✅ Kept Landsat_LC08_016035_20140618.tif (cloud free)
✅ Kept Landsa

❌ Skipped Landsat_LC08_016036_20170914.tif (cloud)
✅ Kept Landsat_LC08_016036_20170930.tif (cloud free)
❌ Skipped Landsat_LC08_016036_20171016.tif (cloud)
✅ Kept Landsat_LC08_016036_20171101.tif (cloud free)
✅ Kept Landsat_LC08_016036_20171117.tif (cloud free)
✅ Kept Landsat_LC08_016036_20171203.tif (cloud free)
❌ Skipped Landsat_LC08_016036_20171219.tif (cloud)
❌ Skipped Landsat_LC08_016036_20180104.tif (cloud)
✅ Kept Landsat_LC08_016036_20180120.tif (cloud free)
✅ Kept Landsat_LC08_016036_20180205.tif (cloud free)
❌ Skipped Landsat_LC08_016036_20180221.tif (cloud)
✅ Kept Landsat_LC08_016036_20180309.tif (cloud free)
❌ Skipped Landsat_LC08_016036_20180325.tif (cloud)
❌ Skipped Landsat_LC08_016036_20180410.tif (cloud)
✅ Kept Landsat_LC08_016036_20180426.tif (cloud free)
❌ Skipped Landsat_LC08_016036_20180512.tif (cloud)
❌ Skipped Landsat_LC08_016036_20180613.tif (cloud)
❌ Skipped Landsat_LC08_016036_20180629.tif (cloud)
❌ Skipped Landsat_LC08_016036_20180715.tif (cloud)
❌ Skipped Lands

In [13]:
input_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\filtered"  
output_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered"  

In [14]:
arcpy.env.overwriteOutput = True  # <-- Add this line

for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Only process .tif files
        raster_path = os.path.join(input_folder, raster_file)

        # Define Raster and extract bands
        raster = Raster(raster_path)
        SWIR1 = arcpy.ia.ExtractBand(raster, band_ids=[4])  # band numbers representing SWIR2 and SWIR1
        SWIR2 = arcpy.ia.ExtractBand(raster, band_ids=[5])  # modify if needed

        # Apply MIRBI formula
        mirbi_raster = (10 * SWIR2) - (9.8 * SWIR1) + 2

        # Define output file path
        output_path = os.path.join(output_folder, f"MIRBI_{raster_file}")

        # Save the output
        mirbi_raster.save(output_path)
        print(f"Processed: {raster_file} → Saved: {output_path}")

print("✅ All MIRBI calculations completed!")

Processed: Landsat_LC08_015036_20130402.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_015036_20130402.tif
Processed: Landsat_LC08_016035_20130717.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20130717.tif
Processed: Landsat_LC08_016035_20131005.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20131005.tif
Processed: Landsat_LC08_016035_20131106.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20131106.tif
Processed: Landsat_LC08_016035_20140109.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20140109.tif
Processed: Landsat_LC08_016035_20140314.tif → Saved: F:\remote sensing

Processed: Landsat_LC08_016035_20220421.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20220421.tif
Processed: Landsat_LC08_016035_20220523.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20220523.tif
Processed: Landsat_LC08_016035_20220624.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20220624.tif
Processed: Landsat_LC08_016035_20221201.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20221201.tif
Processed: Landsat_LC08_016035_20230118.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016035_20230118.tif
Processed: Landsat_LC08_016035_20230307.tif → Saved: F:\remote sensing

Processed: Landsat_LC08_016036_20191225.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016036_20191225.tif
Processed: Landsat_LC08_016036_20200126.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016036_20200126.tif
Processed: Landsat_LC08_016036_20200227.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016036_20200227.tif
Processed: Landsat_LC08_016036_20200602.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016036_20200602.tif
Processed: Landsat_LC08_016036_20200720.tif → Saved: F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered\MIRBI_Landsat_LC08_016036_20200720.tif
Processed: Landsat_LC08_016036_20200906.tif → Saved: F:\remote sensing

In [15]:
# calculate mean

arcpy.env.workspace = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered"   # folder that contains MIBRI
arcpy.env.overwriteOutput = True  


mirbi_rasters = arcpy.ListRasters() 

# Compute mean pixel value across all rasters
mean_mirbi = arcpy.sa.CellStatistics(mirbi_rasters, "MEAN", "DATA")

# Save the output raster
output_raster = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\Mean_MIRBI.tif"  # Change path as needed

mean_mirbi.save(output_raster)




In [16]:
# calculate sd

arcpy.env.workspace = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered"
arcpy.env.overwriteOutput = True  


mirbi_rasters = arcpy.ListRasters() 

# Compute mean pixel value across all rasters
SD_mirbi = arcpy.sa.CellStatistics(mirbi_rasters, "STD", "DATA")

# Save the output raster
output_raster = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\sd_MIRBI.tif"  # Change path as needed
SD_mirbi.save(output_raster)



In [19]:

input_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\MIRBI_filtered"  
output_folder = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\sus_burn_B2"  
mean_mirbi_path = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\Mean_MIRBI.tif"
sd_mirbi_path = r"F:\remote sensing fire detection\VFT remote sensing fire detection\Landsat8_BR\sd_MIRBI.tif"

#gsp_li = arcpy.Point(749137, 3774536)  
#gsp_bi = arcpy.Point(748942, 3774918)
gsp_li = arcpy.Point(749137, 3774536)  
gsp_bi = arcpy.Point(748942, 3774918) 
IA = arcpy.Point(288256,3830171)
CH = arcpy.Point(328503,3848824)
CM = arcpy.Point(319321,3848871)
ME = arcpy.Point(273569,3836308)
B1 = arcpy.Point(663035,3890743)
B2 = arcpy.Point(664053,3890735)
for raster_file in os.listdir(input_folder):
    if raster_file.endswith(".tif"):  # Only process .tif files
        raster_path = os.path.join(input_folder, raster_file)
        raster = Raster(raster_path)
        
    #define variables
        mirbi_value = float(arcpy.GetCellValue_management(raster_path, f"{B2.X} {B2.Y}").getOutput(0)) #modify coord  # must define output as float
        mean_mirbi_value = float(arcpy.GetCellValue_management(mean_mirbi_path, f"{B2.X} {B2.Y}").getOutput(0))
        sd_mirbi_value = float(arcpy.GetCellValue_management(sd_mirbi_path, f"{B2.X} {B2.Y}").getOutput(0))
    
    # Compute absolute deviation from mean
        deviation = abs(mirbi_value - mean_mirbi_value)

  # select those not within 2 sd
        if deviation >= (2 * sd_mirbi_value):
            output_path = os.path.join(output_folder, os.path.basename(raster_path))
            arcpy.CopyRaster_management(raster_path, output_path)
            print(f"✅ Exported: {os.path.basename(raster_path)} (Outlier detected)")
        else:
            print(f"❌ Skipped: {os.path.basename(raster_path)} (Within normal range)")

❌ Skipped: MIRBI_Landsat_LC08_015036_20130402.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20130717.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20131005.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20131106.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140109.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140314.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140517.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140602.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140618.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140704.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140821.tif (Within normal range)
❌ Skipped: MIRBI_Landsat_LC08_016035_20140922.tif (Within normal range)
✅ Exported: MIRBI_Landsat_LC08_016035_20141024.tif (Outlier detected)
✅ Exported: MIRBI_Landsat_LC08_016035_20141211.tif (Outlier detect

❌ Skipped: MIRBI_Landsat_LC08_016036_20241222.tif (Within normal range)


In [ ]:
# LC 8
GSP: 234 downloaded,  134 cloud free, 
Lej: